# APA-DDoS — complete experiment notebook (revision R2)
One run produces every APA-DDoS table and figure in the manuscript.

| Section | Content | Paper item |
|---|---|---|
| 0 | Environment | Section 3.2 |
| 1 | Dataset audit, attribute table, distinct vectors | Tables 2, 2a, 3, 4; Section 4.6.4 |
| 2 | Descriptive statistics, correlation | supplementary |
| 3 | Mutual-information ranking on all 22 attributes | APA MI figure |
| 4–6 | Six classifiers on Raw-22, MI Top-5, MI Top-12 (reports, confusion matrices, ROC) | Table 13, Figure 21, Table 17 |
| 7 | Raw vs Top-5 vs Top-12 | comparison tables/figures |
| 8 | 10-fold CV (SVM excluded for runtime) | CV table |
| 9 | McNemar, all pairs, Holm | statistics table |
| 10 | Timing | timing table |
| 11 | Unconstrained DT / RF / XGBoost on Raw-22 | Section 5.2 |
| 12 | Leakage audit: E0b, E1, E1-Top-5, E2, E3, single-feature tests | Table 14, Figure 22, Sections 4.6.1–4.6.4 |
| 13 | Export (Excel, paper map, zip) | — |

In [1]:
# ===================== CONFIGURATION =====================
import os
DATASET = "APA-DDoS"
DATA_PATH = os.environ.get("APA_PATH", "/kaggle/input/datasets/yashwanthkumbam/apaddos-dataset/APA-DDoS-Dataset/APA-DDoS-Dataset.csv")
OUT = os.environ.get("APA_OUT", "/kaggle/working/APA_DDoS_results")
TARGET = "label"
SEED = 42
N_JOBS = os.cpu_count() or 1
TRAIN_REPEATS = 3
INFER_REPEATS = 10
RUN_CV = True
CV_FOLDS = 10
RUN_SVM_AUDIT = True                 # SVM in the audit experiments (E2 SVM takes ~10 min)
IDENTIFIER_COLS = ["ip.src", "ip.dst", "tcp.srcport", "tcp.dstport", "frame.time", "tcp.seq", "tcp.ack"]
PSH = "tcp.flags.push"
TIME_COL = "frame.time"
FEATURE_SETS = {"RAW": None, "MI Top-5": 5, "MI Top-12": 12}

In [2]:
# ===================== SHARED CODE =====================
# ---------------------------------------------------------------------
# SHARED CODE (identical in the DDoS-SDN and APA-DDoS notebooks)
# ---------------------------------------------------------------------
import os, sys, time, json, platform, itertools, warnings, zipfile
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import chi2, binomtest

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             roc_curve, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "font.size": 10})

DIRS = ["00_environment", "01_dataset_audit", "02_descriptive", "03_mutual_information",
        "04_raw", "05_mi_top5", "06_mi_top12", "07_comparison", "08_cross_validation",
        "09_mcnemar", "10_timing", "11_capacity_check", "12_dataset_specific", "13_export"]
FS_DIR = {"RAW": "04_raw", "MI Top-5": "05_mi_top5", "MI Top-12": "06_mi_top12"}
TABLES = {}          # name -> DataFrame, exported to one Excel file at the end
PAPER_MAP = []       # (output file, paper item, description)


def setup_dirs():
    for d in DIRS:
        os.makedirs(os.path.join(OUT, d), exist_ok=True)


def p(d, name):
    return os.path.join(OUT, d, name)


def save_table(df, d, name, paper="", desc="", index=False):
    path = p(d, name + ".csv")
    df.to_csv(path, index=index)
    TABLES[name[:31]] = df.reset_index() if index else df
    PAPER_MAP.append((os.path.relpath(path, OUT), paper, desc))
    return path


def save_fig(fig, d, name, paper="", desc=""):
    path = p(d, name + ".png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    PAPER_MAP.append((os.path.relpath(path, OUT), paper, desc))
    return path


# ---------------- environment ----------------
def environment():
    cpu = platform.processor() or ""
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    cpu = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
    ram = ""
    try:
        with open("/proc/meminfo") as f:
            ram = f"{int(f.readline().split()[1]) / 1024**2:.1f} GB"
    except Exception:
        pass
    env = pd.DataFrame([
        ("CPU", cpu), ("Logical CPUs", os.cpu_count()), ("RAM", ram), ("OS", platform.platform()),
        ("Python", platform.python_version()), ("scikit-learn", sklearn.__version__),
        ("XGBoost", xgboost.__version__), ("NumPy", np.__version__), ("pandas", pd.__version__),
        ("Threads per model (n_jobs)", N_JOBS), ("Device", "CPU only; no GPU used by any classifier"),
        ("Random seed", SEED), ("Training-time repeats", TRAIN_REPEATS),
        ("Inference-time repeats", INFER_REPEATS),
    ], columns=["Item", "Value"])
    save_table(env, "00_environment", "environment", "Section 3.2 / Table of setup",
               "Hardware and software used for every timing result")
    print(env.to_string(index=False))
    return env


# ---------------- models ----------------
def models_main():
    """Section 3.2 hyper-parameters (fixed a priori)."""
    lr_solver = "liblinear" if N_CLASSES == 2 else "lbfgs"
    return {
        "Logistic Regression": LogisticRegression(solver=lr_solver, C=0.03, max_iter=1000,
                                                  random_state=SEED),
        "SVM": SVC(kernel="rbf", C=1.0, probability=False, random_state=SEED),
        "KNN": KNeighborsClassifier(n_neighbors=5, metric="manhattan", weights="distance",
                                    n_jobs=N_JOBS),
        "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=8, max_leaf_nodes=11,
                                                random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=20, criterion="entropy", max_depth=10,
                                                min_samples_split=100, min_samples_leaf=20,
                                                max_features="sqrt", random_state=SEED,
                                                n_jobs=N_JOBS),
        "XGBoost": XGBClassifier(eval_metric="logloss" if N_CLASSES == 2 else "mlogloss",
                                 tree_method="hist", random_state=SEED, n_jobs=N_JOBS),
    }


def models_unconstrained():
    return {
        "Decision Tree (unconstrained)": DecisionTreeClassifier(random_state=SEED),
        "Random Forest (unconstrained)": RandomForestClassifier(n_estimators=200, criterion="entropy",
                                                                max_features="sqrt", random_state=SEED,
                                                                n_jobs=N_JOBS),
        "XGBoost (500 trees, depth 10)": XGBClassifier(n_estimators=500, max_depth=10,
                                                       eval_metric="logloss" if N_CLASSES == 2 else "mlogloss",
                                                       tree_method="hist", random_state=SEED, n_jobs=N_JOBS),
    }


# ---------------- pipeline ----------------
class MISelector(BaseEstimator, TransformerMixin):
    """Mutual information (sklearn mutual_info_classif: k-NN estimator, n_neighbors=3,
    all inputs treated as continuous after encoding, fixed random_state).
    Fitted on training data only; ties broken by column order (stable sort)."""
    def __init__(self, k=5, random_state=42):
        self.k, self.random_state = k, random_state
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.scores_ = mutual_info_classif(X, y, random_state=self.random_state)
        order = np.argsort(-self.scores_, kind="stable")
        self.indices_ = np.sort(order[: min(self.k, X.shape[1])])
        return self
    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.indices_]


def make_pipeline(num_cols, cat_cols, model=None, k=None):
    """impute -> ordinal-encode (unseen categories -> -1) -> [MI top-k] -> scale -> model.
    Every step is fitted on the training data (or training fold) only."""
    tr = []
    if num_cols:
        tr.append(("num", SimpleImputer(strategy="median"), list(num_cols)))
    if cat_cols:
        tr.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                           unknown_value=-1))]), list(cat_cols)))
    steps = [("prep", ColumnTransformer(tr, verbose_feature_names_out=False))]
    if k is not None:
        steps.append(("mi", MISelector(k=k, random_state=SEED)))
    steps.append(("scale", StandardScaler()))
    if model is not None:
        steps.append(("clf", model))
    return Pipeline(steps)


def selected_names(prep_pipe, num_cols, cat_cols):
    names = list(num_cols) + list(cat_cols)
    if "mi" in prep_pipe.named_steps:
        return [names[i] for i in prep_pipe.named_steps["mi"].indices_]
    return names


def model_scores(clf, Xt):
    """Class scores used for ROC/PR: probabilities if available, otherwise decision values."""
    if hasattr(clf, "predict_proba"):
        s = clf.predict_proba(Xt)
    else:
        s = clf.decision_function(Xt)
    if N_CLASSES == 2:
        return s[:, 1] if s.ndim == 2 else s
    return s


# ---------------- metrics ----------------
def ovr_auc(y, s):
    return float(np.mean([roc_auc_score((y == k).astype(int), s[:, k]) for k in range(N_CLASSES)]))


def ovr_ap(y, s):
    return float(np.mean([average_precision_score((y == k).astype(int), s[:, k]) for k in range(N_CLASSES)]))


def compute_metrics(y, pred, s):
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    if N_CLASSES == 2:
        tn, fp, fn, tp = cm.ravel()
        return {"Accuracy (%)": accuracy_score(y, pred) * 100,
                "Precision (%)": precision_score(y, pred, zero_division=0) * 100,
                "Recall (%)": recall_score(y, pred, zero_division=0) * 100,
                "Specificity (%)": tn / (tn + fp) * 100 if tn + fp else np.nan,
                "F1 (%)": f1_score(y, pred, zero_division=0) * 100,
                "Weighted F1 (%)": f1_score(y, pred, average="weighted") * 100,
                "ROC-AUC": roc_auc_score(y, s), "PR-AUC": average_precision_score(y, s),
                "TN": tn, "FP": fp, "FN": fn, "TP": tp}
    spec = []
    for k in range(N_CLASSES):
        tn = cm.sum() - cm[k, :].sum() - cm[:, k].sum() + cm[k, k]; fp = cm[:, k].sum() - cm[k, k]
        spec.append(tn / (tn + fp) if tn + fp else np.nan)
    out = {"Accuracy (%)": accuracy_score(y, pred) * 100,
           "Macro precision (%)": precision_score(y, pred, average="macro", zero_division=0) * 100,
           "Macro recall (%)": recall_score(y, pred, average="macro", zero_division=0) * 100,
           "Macro specificity (%)": np.nanmean(spec) * 100,
           "Macro F1 (%)": f1_score(y, pred, average="macro", zero_division=0) * 100,
           "Micro F1 (%)": f1_score(y, pred, average="micro", zero_division=0) * 100,
           "Weighted F1 (%)": f1_score(y, pred, average="weighted", zero_division=0) * 100,
           "ROC-AUC (macro OvR)": ovr_auc(y, s), "PR-AUC (macro OvR)": ovr_ap(y, s)}
    for k, name in enumerate(CLASS_NAMES):
        out[f"Recall {name} (%)"] = cm[k, k] / cm[k, :].sum() * 100 if cm[k, :].sum() else np.nan
    return out


def acc_key():
    return "Accuracy (%)"


def f1_key():
    return "F1 (%)" if N_CLASSES == 2 else "Macro F1 (%)"


def auc_key():
    return "ROC-AUC" if N_CLASSES == 2 else "ROC-AUC (macro OvR)"


def mcnemar(y, p1, p2):
    a, b_ = (p1 == y), (p2 == y)
    b = int(np.sum(a & ~b_)); c = int(np.sum(~a & b_))
    if b + c == 0:
        return b, c, np.nan, 1.0, "no discordant pairs"
    if b + c < 25:
        return b, c, np.nan, float(binomtest(b, b + c, 0.5).pvalue), "exact binomial"
    st = (abs(b - c) - 1) ** 2 / (b + c)
    return b, c, st, float(chi2.sf(st, 1)), "chi-square with continuity correction"


def holm(pvals):
    pv = np.asarray(pvals, float); m = len(pv); order = np.argsort(pv)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * pv[i]); adj[i] = min(run, 1.0)
    return adj


# ---------------- figures ----------------
def fig_cm_single(cm, title):
    fig, ax = plt.subplots(figsize=(4.4, 3.9))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=max(cm.max(), 1))
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=10, fontweight="bold",
                    color="white" if cm[i, j] > cm.max() / 2 else "#08306b")
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=20, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title, fontsize=10, fontweight="bold")
    fig.tight_layout()
    return fig


def fig_cm_grid(cms, suptitle):
    n = len(cms); cols = 3; rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 4.1 * rows))
    axes = np.atleast_1d(axes).ravel()
    letters = "abcdefghij"
    for i, (ax, (name, cm)) in enumerate(zip(axes, cms.items())):
        acc = np.trace(cm) / cm.sum() * 100
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=max(cm.max(), 1))
        for a in range(cm.shape[0]):
            for b in range(cm.shape[1]):
                ax.text(b, a, f"{cm[a, b]:,}", ha="center", va="center", fontsize=10, fontweight="bold",
                        color="white" if cm[a, b] > cm.max() / 2 else "#08306b")
        ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=20, ha="right", fontsize=8)
        ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        ax.set_title(f"({letters[i]}) {name}\nAcc = {acc:.2f}%", fontsize=10, fontweight="bold")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=12); fig.tight_layout()
    return fig


def fig_roc(curves, title):
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    for name, (fpr, tpr, auc) in curves.items():
        ax.plot(fpr, tpr, lw=1.8, label=f"{name} (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(title, fontsize=11); ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def roc_points(y, s):
    if N_CLASSES == 2:
        fpr, tpr, _ = roc_curve(y, s); return fpr, tpr
    Y = np.eye(N_CLASSES)[y]                      # micro-average OvR curve for display
    fpr, tpr, _ = roc_curve(Y.ravel(), np.asarray(s).ravel()); return fpr, tpr


def fig_grouped_bars(df, value, ylabel, title, order, log=False, annotate=True, legend_title="Feature set"):
    piv = df.pivot_table(index="Model", columns="Feature set", values=value, sort=False)
    piv = piv[[c for c in order if c in piv.columns]]
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    piv.plot(kind="bar", ax=ax, logy=log, edgecolor="black", width=0.8)
    if annotate:
        for cont in ax.containers:
            ax.bar_label(cont, fmt="%.2f" if not log else "%.3g", fontsize=7, rotation=90, padding=2)
    ax.set_ylabel(ylabel); ax.set_xlabel(""); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
    ax.legend(title=legend_title, fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right"); fig.tight_layout()
    return fig


# ---------------- evaluation on a fixed split ----------------
def evaluate(X, y, tr, te, num_cols, cat_cols, model_dict, feature_sets, tag, out_dirs,
             timing=True, save_reports=True, figures=True):
    """Fit every model on the training indices, evaluate once on the test indices.
    Saves classification reports, confusion matrices and ROC curves per feature set."""
    rows, preds, selected = [], {}, {}
    yte = np.asarray(y)[te]; ytr = np.asarray(y)[tr]
    for fs, k in feature_sets.items():
        d = out_dirs[fs]
        prep = make_pipeline(num_cols, cat_cols, None, k)
        t0 = time.perf_counter(); prep.fit(X.iloc[tr], ytr); t_fit_prep = time.perf_counter() - t0
        t0 = time.perf_counter(); Xtr_t = prep.transform(X.iloc[tr]); t_tr_tf = time.perf_counter() - t0
        t0 = time.perf_counter(); Xte_t = prep.transform(X.iloc[te]); t_te_tf = time.perf_counter() - t0
        selected[fs] = selected_names(prep, num_cols, cat_cols)
        fslug = fs.replace(" ", "").replace("-", "")
        cms, curves = {}, {}
        for name, base in model_dict.items():
            reps = 1 if (not timing or name == "SVM") else TRAIN_REPEATS
            fit_t = []
            for _ in range(reps):
                clf = clone(base)
                t0 = time.perf_counter(); clf.fit(Xtr_t, ytr); fit_t.append(time.perf_counter() - t0)
            inf_t = []
            for _ in range(INFER_REPEATS if timing else 1):
                t0 = time.perf_counter(); pred = clf.predict(Xte_t); inf_t.append(time.perf_counter() - t0)
            s = model_scores(clf, Xte_t)
            m = compute_metrics(yte, pred, s)
            n = len(te)
            row = {"Split": tag, "Feature set": fs, "Model": name, "N features": len(selected[fs]), **m,
                   "Preprocessing+MI fit (s)": t_fit_prep, "Transform train+test (s)": t_tr_tf + t_te_tf,
                   "Training time mean (s)": np.mean(fit_t),
                   "Training time SD (s)": np.std(fit_t, ddof=1) if len(fit_t) > 1 else np.nan,
                   "Training repeats": len(fit_t),
                   "Inference time mean (s)": np.mean(inf_t),
                   "Inference time SD (s)": np.std(inf_t, ddof=1) if len(inf_t) > 1 else np.nan,
                   "Latency (ms/sample)": np.mean(inf_t) / n * 1000,
                   "Throughput (samples/s)": n / np.mean(inf_t), "Test samples": n}
            rows.append(row); preds[(fs, name)] = pred
            cm = confusion_matrix(yte, pred, labels=list(range(N_CLASSES))); cms[name] = cm
            fpr, tpr = roc_points(yte, s); curves[name] = (fpr, tpr, m[auc_key()])
            safe = name.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
            if save_reports:
                rep = classification_report(yte, pred, labels=list(range(N_CLASSES)), target_names=CLASS_NAMES,
                                            digits=4, zero_division=0)
                with open(p(d, f"classification_report_{tag}_{fslug}_{safe}.txt"), "w") as fh:
                    fh.write(f"{DATASET} | {tag} | {fs} | {name}\n\n{rep}")
                pd.DataFrame(classification_report(yte, pred, labels=list(range(N_CLASSES)),
                                                   target_names=CLASS_NAMES, output_dict=True,
                                                   zero_division=0)).T.round(4).to_csv(
                    p(d, f"classification_report_{tag}_{fslug}_{safe}.csv"))
                if figures:
                    save_fig(fig_cm_single(cm, f"{name} — {fs}"), d, f"cm_{tag}_{fslug}_{safe}",
                             "", f"Confusion matrix {name}, {fs}, {tag}")
            print(f"  [{tag}] {fs:<10s} {name:<30s} acc {m[acc_key()]:8.4f}  F1 {m[f1_key()]:8.4f}  "
                  f"AUC {m[auc_key()]:.4f}  train {np.mean(fit_t):8.3f}s  infer {np.mean(inf_t)*1000:9.2f}ms")
        if figures:
            save_fig(fig_cm_grid(cms, f"{DATASET} — {fs} — {tag} (held-out test set)"), d,
                     f"fig_cm_grid_{tag}_{fslug}", f"Confusion-matrix figure ({fs}, {tag})", f"All models, {fs}, {tag}")
            save_fig(fig_roc(curves, f"ROC curves — {DATASET}, {fs} ({tag})" +
                             ("" if N_CLASSES == 2 else "\nmicro-average curve; legend AUC = macro one-vs-rest")),
                     d, f"fig_roc_{tag}_{fslug}", f"ROC figure ({fs}, {tag})", f"All models, {fs}, {tag}")
    return pd.DataFrame(rows), preds, selected


# ---------------- 10-fold CV ----------------
def cross_validate_models(X, y, tr, num_cols, cat_cols, model_dict, feature_sets, folds=10):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    Xtr = X.iloc[tr].reset_index(drop=True); ytr = np.asarray(y)[tr]
    rows = []
    for fs, k in feature_sets.items():
        for name, base in model_dict.items():
            acc, prec, rec, f1w, auc = [], [], [], [], []
            for a, b in skf.split(Xtr, ytr):
                pipe = make_pipeline(num_cols, cat_cols, clone(base), k).fit(Xtr.iloc[a], ytr[a])
                pr = pipe.predict(Xtr.iloc[b]); s = model_scores(pipe.named_steps["clf"], Pipeline(pipe.steps[:-1]).transform(Xtr.iloc[b]))
                yt = ytr[b]
                acc.append(accuracy_score(yt, pr) * 100)
                prec.append(precision_score(yt, pr, average="weighted", zero_division=0) * 100)
                rec.append(recall_score(yt, pr, average="weighted", zero_division=0) * 100)
                f1w.append(f1_score(yt, pr, average="weighted", zero_division=0) * 100)
                auc.append((roc_auc_score(yt, s) if N_CLASSES == 2 else ovr_auc(yt, s)) * 100)
            rows.append({"Feature set": fs, "Model": name, "Folds": folds,
                         "Accuracy mean (%)": np.mean(acc), "Accuracy SD": np.std(acc, ddof=1),
                         "Weighted precision mean (%)": np.mean(prec), "Weighted recall mean (%)": np.mean(rec),
                         "Weighted F1 mean (%)": np.mean(f1w), "Weighted F1 SD": np.std(f1w, ddof=1),
                         "ROC-AUC mean (%)": np.mean(auc), "ROC-AUC SD": np.std(auc, ddof=1)})
            print(f"  CV {fs:<10s} {name:<22s} {np.mean(acc):7.3f} ± {np.std(acc, ddof=1):.3f}")
    return pd.DataFrame(rows)


def cv_paper_table(cv, fs):
    d = cv[cv["Feature set"] == fs]
    return pd.DataFrame({
        "Model": d["Model"],
        "Accuracy (Mean ± SD)": [f"{a:.2f} ± {b:.2f}" for a, b in zip(d["Accuracy mean (%)"], d["Accuracy SD"])],
        "Weighted Precision (%)": d["Weighted precision mean (%)"].round(2).values,
        "Weighted Recall (%)": d["Weighted recall mean (%)"].round(2).values,
        "Weighted F1 (Mean ± SD)": [f"{a:.2f} ± {b:.2f}" for a, b in zip(d["Weighted F1 mean (%)"], d["Weighted F1 SD"])],
        "ROC-AUC (%)": d["ROC-AUC mean (%)"].round(2).values})


# ---------------- McNemar (all pairs, Holm) ----------------
def mcnemar_all(y_te, preds, feature_sets, model_names):
    rows = []
    for fs in feature_sets:
        tmp = []
        for m1, m2 in itertools.combinations(model_names, 2):
            b, c, st, pv, test = mcnemar(y_te, preds[(fs, m1)], preds[(fs, m2)])
            tmp.append({"Feature set": fs, "Model 1": m1, "Model 2": m2,
                        "b (Model 1 correct, Model 2 wrong)": b, "c (Model 1 wrong, Model 2 correct)": c,
                        "Statistic": st, "p-value": pv, "Test": test})
        adj = holm([t["p-value"] for t in tmp])
        for t, a in zip(tmp, adj):
            t["p-value (Holm)"] = a; t["Significant at 0.05 (Holm)"] = "Yes" if a < 0.05 else "No"
        rows += tmp
    return pd.DataFrame(rows)


# ---------------- comparison figures (Raw vs MI) ----------------
def comparison_outputs(res, fs_new, tag_fig_acc, tag_fig_f1, tag_fig_cost, tag_fig_heat, paper_tab):
    a = res[res["Feature set"] == "RAW"].set_index("Model")
    b = res[res["Feature set"] == fs_new].set_index("Model")
    tab = pd.DataFrame({
        "Accuracy Raw (%)": a[acc_key()], f"Accuracy {fs_new} (%)": b[acc_key()],
        "Δ Accuracy (pp)": b[acc_key()] - a[acc_key()],
        "Weighted F1 Raw (%)": a["Weighted F1 (%)"], f"Weighted F1 {fs_new} (%)": b["Weighted F1 (%)"],
        "Δ Weighted F1 (pp)": b["Weighted F1 (%)"] - a["Weighted F1 (%)"],
        "Training Raw (s)": a["Training time mean (s)"], f"Training {fs_new} (s)": b["Training time mean (s)"],
        "Inference Raw (ms)": a["Inference time mean (s)"] * 1000,
        f"Inference {fs_new} (ms)": b["Inference time mean (s)"] * 1000,
        "Preprocessing+MI Raw (s)": a["Preprocessing+MI fit (s)"],
        f"Preprocessing+MI {fs_new} (s)": b["Preprocessing+MI fit (s)"]}).round(4)
    save_table(tab, "07_comparison", f"raw_vs_{fs_new.replace(' ', '').replace('-', '')}", paper_tab,
               f"Raw vs {fs_new}: accuracy, F1, training, inference, preprocessing", index=True)
    models = list(tab.index); x = np.arange(len(models)); w = 0.38
    # accuracy + time
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    for ax, (c1, c2, lab, log) in zip(axes, [("Accuracy Raw (%)", f"Accuracy {fs_new} (%)", "Accuracy (%)", False),
                                             ("Training Raw (s)", f"Training {fs_new} (s)", "Training time (s, log)", True)]):
        r1 = ax.bar(x - w / 2, tab[c1], w, label="Raw", edgecolor="black")
        r2 = ax.bar(x + w / 2, tab[c2], w, label=fs_new, edgecolor="black")
        ax.bar_label(r1, fmt="%.2f" if not log else "%.3g", fontsize=7, rotation=90, padding=2)
        ax.bar_label(r2, fmt="%.2f" if not log else "%.3g", fontsize=7, rotation=90, padding=2)
        if log: ax.set_yscale("log")
        else: ax.set_ylim(max(0, min(tab[c1].min(), tab[c2].min()) - 10), 104)
        ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, ha="right"); ax.set_ylabel(lab)
        ax.legend(); ax.grid(axis="y", alpha=0.3)
    axes[0].set_title(f"Accuracy: Raw vs {fs_new}"); axes[1].set_title(f"Training time: Raw vs {fs_new}")
    fig.tight_layout(); save_fig(fig, "07_comparison", f"fig_accuracy_time_{fs_new.replace(' ', '')}", tag_fig_acc,
                                 f"Accuracy and training time, Raw vs {fs_new}")
    # F1 change
    fig, ax = plt.subplots(figsize=(9, 4.4))
    dlt = tab["Δ Weighted F1 (pp)"]
    bars = ax.bar(models, dlt, color=["#2ca02c" if v >= 0 else "#d62728" for v in dlt], edgecolor="black")
    ax.bar_label(bars, fmt="%+.2f", fontsize=8); ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("Δ weighted F1 (percentage points)")
    ax.set_title(f"Change in weighted-average F1-score: {fs_new} − Raw"); ax.grid(axis="y", alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right"); fig.tight_layout()
    save_fig(fig, "07_comparison", f"fig_f1_change_{fs_new.replace(' ', '')}", tag_fig_f1, f"Δ weighted F1, {fs_new}")
    # computational cost: stacked preprocessing / training / inference
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for off, src, lab in [(-w / 2, a, "Raw"), (w / 2, b, fs_new)]:
        p1 = src["Preprocessing+MI fit (s)"].values; t1 = src["Training time mean (s)"].values
        i1 = src["Inference time mean (s)"].values
        ax.bar(x + off, p1, w, label=f"{lab}: preprocessing+MI", edgecolor="black", alpha=0.6)
        ax.bar(x + off, t1, w, bottom=p1, label=f"{lab}: training", edgecolor="black")
        ax.bar(x + off, i1, w, bottom=p1 + t1, label=f"{lab}: inference", edgecolor="black", hatch="//")
    ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, ha="right")
    ax.set_ylabel("Time (s, log scale)"); ax.set_title(f"Computational cost: Raw vs {fs_new}")
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left"); ax.grid(axis="y", alpha=0.3); fig.tight_layout()
    save_fig(fig, "07_comparison", f"fig_cost_{fs_new.replace(' ', '')}", tag_fig_cost,
             f"Preprocessing+MI, training and inference time, Raw vs {fs_new}")
    # heatmap
    hm = pd.DataFrame({"Acc Raw": tab["Accuracy Raw (%)"], f"Acc {fs_new}": tab[f"Accuracy {fs_new} (%)"],
                       "Wtd F1 Raw": tab["Weighted F1 Raw (%)"], f"Wtd F1 {fs_new}": tab[f"Weighted F1 {fs_new} (%)"]}).T
    fig, ax = plt.subplots(figsize=(10, 3.6))
    im = ax.imshow(hm.values, cmap="YlGnBu", aspect="auto", vmin=min(60, hm.values.min()), vmax=100)
    for i in range(hm.shape[0]):
        for j in range(hm.shape[1]):
            ax.text(j, i, f"{hm.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns, rotation=20, ha="right")
    ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels(hm.index); fig.colorbar(im, ax=ax, label="%")
    ax.set_title(f"Accuracy and weighted F1 (%): Raw vs {fs_new}"); fig.tight_layout()
    save_fig(fig, "07_comparison", f"fig_heatmap_{fs_new.replace(' ', '')}", tag_fig_heat,
             f"Heatmap of accuracy and weighted F1, Raw vs {fs_new}")
    return tab


def cv_figure(cv, order, paper):
    fig, axes = plt.subplots(1, len(order), figsize=(5.2 * len(order), 4.6), sharey=False)
    for ax, fs in zip(np.atleast_1d(axes), order):
        d = cv[cv["Feature set"] == fs]
        bars = ax.bar(d["Model"], d["Accuracy mean (%)"], yerr=d["Accuracy SD"], capsize=4, edgecolor="black")
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=3)
        ax.set_ylim(max(0, d["Accuracy mean (%)"].min() - 8), 101); ax.set_title(f"10-fold CV accuracy — {fs}")
        ax.set_ylabel("Accuracy (%)"); plt.setp(ax.get_xticklabels(), rotation=25, ha="right"); ax.grid(axis="y", alpha=0.3)
    fig.tight_layout(); save_fig(fig, "08_cross_validation", "fig_cv_accuracy", paper, "10-fold CV accuracy (mean ± SD)")


def timing_outputs(res, order, paper_tab, paper_lat, paper_thr):
    t = res[["Feature set", "Model", "N features", "Preprocessing+MI fit (s)", "Transform train+test (s)",
             "Training time mean (s)", "Training time SD (s)", "Training repeats", "Inference time mean (s)",
             "Inference time SD (s)", "Latency (ms/sample)", "Throughput (samples/s)"]].round(6)
    save_table(t, "10_timing", "timing", paper_tab, "Preprocessing, training and inference time (mean ± SD), latency, throughput")
    save_fig(fig_grouped_bars(res, "Latency (ms/sample)", "Latency (ms/sample, log)", "Inference latency", order, log=True),
             "10_timing", "fig_latency", paper_lat, "Inference latency by feature set")
    save_fig(fig_grouped_bars(res, "Throughput (samples/s)", "Throughput (samples/s, log)", "Inference throughput", order, log=True),
             "10_timing", "fig_throughput", paper_thr, "Inference throughput by feature set")
    save_fig(fig_grouped_bars(res, "Training time mean (s)", "Training time (s, log)", "Training time", order, log=True),
             "10_timing", "fig_training_time", "", "Training time by feature set")


def mi_outputs(X, y, tr, num_cols, cat_cols, paper_prefix):
    prep = make_pipeline(num_cols, cat_cols, None, None)
    Xt = Pipeline(prep.steps[:-1]).fit(X.iloc[tr], np.asarray(y)[tr]).transform(X.iloc[tr])
    mi = mutual_info_classif(np.asarray(Xt, float), np.asarray(y)[tr], random_state=SEED)
    df = pd.DataFrame({"Feature": list(num_cols) + list(cat_cols), "MI score": mi})
    df = df.sort_values("MI score", ascending=False, kind="stable").reset_index(drop=True)
    df["Rank"] = np.arange(1, len(df) + 1)
    df["In Top-5"] = df["Rank"] <= 5; df["In Top-12"] = df["Rank"] <= 12
    save_table(df, "03_mutual_information", "mi_ranking", f"{paper_prefix} MI ranking table",
               "MI score of every input feature (training data)")
    fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(df)), 4.6))
    cols = ["#1f77b4" if r <= 5 else "#6baed6" if r <= 12 else "#c6dbef" for r in df["Rank"]]
    ax.bar(df["Feature"], df["MI score"], color=cols, edgecolor="black")
    ax.set_ylabel("MI score"); ax.set_title("Mutual information of each feature (training data; dark = Top-5, mid = Top-12)")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right"); ax.grid(axis="y", alpha=0.3); fig.tight_layout()
    save_fig(fig, "03_mutual_information", "fig_mi_scores", f"{paper_prefix} Figure 3", "MI scores, all features")
    fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(df)), 1.9))
    im = ax.imshow(df[["MI score"]].T.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(df))); ax.set_xticklabels(df["Feature"], rotation=45, ha="right"); ax.set_yticks([])
    for j, v in enumerate(df["MI score"]):
        ax.text(j, 0, f"{v:.3f}", ha="center", va="center", fontsize=7, color="white" if v < df["MI score"].max() * 0.6 else "black")
    fig.colorbar(im, ax=ax, fraction=0.03); ax.set_title("Feature importance heatmap (MI score)"); fig.tight_layout()
    save_fig(fig, "03_mutual_information", "fig_mi_heatmap", f"{paper_prefix} Figure 4", "MI heatmap")
    for name, part, title, paper in [("fig_mi_top10", df.head(10), "Top-10 features by MI", "Figure 5"),
                                     ("fig_mi_least", df.tail(max(1, len(df) - 12)).iloc[::-1],
                                      "Features outside the Top-12 (least informative)", "Figure 6")]:
        fig, ax = plt.subplots(figsize=(7, 0.35 * len(part) + 1.2))
        ax.barh(part["Feature"][::-1] if name == "fig_mi_top10" else part["Feature"],
                part["MI score"][::-1] if name == "fig_mi_top10" else part["MI score"], edgecolor="black")
        ax.set_xlabel("MI score"); ax.set_title(title); ax.grid(axis="x", alpha=0.3); fig.tight_layout()
        save_fig(fig, "03_mutual_information", name, f"{paper_prefix} {paper}", title)
    return df


def descriptive_outputs(X, num_cols, tr, paper_tab, paper_fig):
    desc = X[num_cols].describe().T[["mean", "std", "25%", "50%", "75%", "max"]].round(2)
    save_table(desc, "02_descriptive", "descriptive_statistics", paper_tab, "Descriptive statistics of numeric inputs", index=True)
    nonconst = [c for c in num_cols if X[c].iloc[tr].nunique() > 1]
    corr = X[nonconst].iloc[tr].corr()
    fig, ax = plt.subplots(figsize=(0.55 * len(nonconst) + 3, 0.5 * len(nonconst) + 2))
    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    for i in range(len(nonconst)):
        for j in range(len(nonconst)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
    ax.set_xticks(range(len(nonconst))); ax.set_xticklabels(nonconst, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(nonconst))); ax.set_yticklabels(nonconst, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.04); ax.set_title("Pearson correlation of numeric inputs (training data)")
    fig.tight_layout(); save_fig(fig, "02_descriptive", "fig_correlation", paper_fig, "Correlation heatmap")
    save_table(corr.round(4), "02_descriptive", "correlation_matrix", "", "Correlation values", index=True)


def vector_audit(X, y, tr, te, label=""):
    v = X.astype(str).agg("|".join, axis=1)
    g = pd.DataFrame({"v": v.values, "y": np.asarray(y)})
    nlab = g.groupby("v")["y"].nunique()
    major = g.groupby("v")["y"].agg(lambda s: s.value_counts().iloc[0]).sum()
    trv = set(v.iloc[tr])
    return {"Feature set": label, "Features": X.shape[1], "Records": len(X),
            "Duplicate records (features + label)": int(pd.concat([X.reset_index(drop=True), pd.Series(np.asarray(y), name='__y')], axis=1).duplicated().sum()),
            "Duplicate records (features only)": int(X.duplicated().sum()),
            "Distinct feature vectors": int(v.nunique()),
            "Vectors with more than one label": int((nlab > 1).sum()),
            "Accuracy ceiling (%)": round(major / len(g) * 100, 4),
            "Test records whose vector occurs in training (%)": round(v.iloc[te].isin(trv).mean() * 100, 4)}


def export_all():
    pm = pd.DataFrame(PAPER_MAP, columns=["Output file", "Paper item", "Description"])
    pm.to_csv(p("13_export", "paper_map.csv"), index=False)
    with pd.ExcelWriter(p("13_export", f"{DATASET.replace(' ', '_')}_all_tables.xlsx")) as xw:
        for name, df in TABLES.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
        pm.to_excel(xw, sheet_name="paper_map", index=False)
    zpath = os.path.join(os.path.dirname(os.path.abspath(OUT)) or ".", f"{os.path.basename(OUT)}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(OUT):
            for f in files:
                full = os.path.join(root, f); z.write(full, os.path.relpath(full, os.path.dirname(os.path.abspath(OUT))))
    print(f"\nExported {len(TABLES)} tables and {sum(1 for r in PAPER_MAP if r[0].endswith('.png'))} figures.")
    print(f"Paper map: {p('13_export', 'paper_map.csv')}\nZip for download: {zpath}")

## 0. Environment

In [3]:
setup_dirs()
ENV = environment()

                      Item                                   Value
                       CPU          Intel(R) Xeon(R) CPU @ 2.20GHz
              Logical CPUs                                       4
                       RAM                                 31.3 GB
                        OS    Linux-6.18.48+-x86_64-with-glibc2.39
                    Python                                 3.13.15
              scikit-learn                                   1.6.1
                   XGBoost                                   3.4.1
                     NumPy                                   2.1.3
                    pandas                                   2.3.3
Threads per model (n_jobs)                                       4
                    Device CPU only; no GPU used by any classifier
               Random seed                                      42
     Training-time repeats                                       3
    Inference-time repeats                                    

## 1. Dataset audit and attribute table

In [4]:
raw = pd.read_csv(DATA_PATH, low_memory=False)
raw.columns = raw.columns.str.strip().str.lower()
print("File:", raw.shape, "\nColumns:", list(raw.columns))
n_missing = int(raw.isna().any(axis=1).sum())
df = raw.dropna().reset_index(drop=True)
LE = LabelEncoder(); y = LE.fit_transform(df[TARGET]); CLASS_NAMES = list(LE.classes_); N_CLASSES = len(CLASS_NAMES)
X = df.drop(columns=[TARGET])                                   # Raw-22: every attribute
cat_cols = list(X.select_dtypes(include=["object", "category", "bool"]).columns)
num_cols = [c for c in X.columns if c not in cat_cols]
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
y_te = y[te]
print(f"Classes {CLASS_NAMES} | records {len(df):,} | train {len(tr):,} | test {len(te):,} | attributes {X.shape[1]}")

const_train = [c for c in X.columns if X[c].iloc[tr].nunique() <= 1]
CLEAN = [c for c in X.columns if c not in IDENTIFIER_COLS and c not in const_train]
E2_COLS = [c for c in CLEAN if c != PSH]
att = pd.DataFrame({"Attribute": X.columns,
                    "Type": [str(X[c].dtype) for c in X.columns],
                    "Distinct values (train)": [X[c].iloc[tr].nunique() for c in X.columns]})
att["Identifier"] = att["Attribute"].isin(IDENTIFIER_COLS)
att["Constant on train"] = att["Attribute"].isin(const_train)
att["Treatment"] = ["Excluded (identifier" + ("; constant)" if c in const_train else ")") if c in IDENTIFIER_COLS
                    else "Excluded (constant)" if c in const_train
                    else "Used (removed in E2)" if c == PSH else "Used" for c in X.columns]
save_table(att, "01_dataset_audit", "attribute_table", "Table 2a", "All 22 attributes and their treatment")
print(att.to_string(index=False))

counts = pd.Series(y).value_counts().sort_index()
cls = pd.DataFrame({"Class": CLASS_NAMES, "All": counts.values,
                    "Train": pd.Series(y[tr]).value_counts().sort_index().values,
                    "Test": pd.Series(y_te).value_counts().sort_index().values})
save_table(cls, "01_dataset_audit", "class_distribution", "Tables 3-4", "Class counts, train/test split")

audit = pd.DataFrame([("Records in file", len(raw)), ("Records with missing values (removed)", n_missing),
                      ("Records used", len(df)), ("Exact duplicate rows (all columns)", int(raw.duplicated().sum()))],
                     columns=["Quantity", "Value"])
save_table(audit, "01_dataset_audit", "dataset_audit", "New Table (dataset audit)", "Missing values and duplicates")
VEC = pd.DataFrame([vector_audit(X, y, tr, te, "Raw-22 (all attributes)"),
                    vector_audit(X[CLEAN], y, tr, te, f"E1 clean ({len(CLEAN)} features)"),
                    vector_audit(X[E2_COLS], y, tr, te, f"E2 clean without PSH ({len(E2_COLS)} features)")])
save_table(VEC, "01_dataset_audit", "distinct_vectors", "Section 4.6.4 / audit table",
           "Distinct vectors, vectors with more than one label, accuracy ceiling, train/test overlap")
print(audit.to_string(index=False)); print(VEC.to_string(index=False))

File: (151200, 23) 
Columns: ['ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'ip.proto', 'frame.len', 'tcp.flags.syn', 'tcp.flags.reset', 'tcp.flags.push', 'tcp.flags.ack', 'ip.flags.mf', 'ip.flags.df', 'ip.flags.rb', 'tcp.seq', 'tcp.ack', 'frame.time', 'packets', 'bytes', 'tx packets', 'tx bytes', 'rx packets', 'rx bytes', 'label']
Classes ['Benign', 'DDoS-ACK', 'DDoS-PSH-ACK'] | records 151,200 | train 105,840 | test 45,360 | attributes 22
      Attribute   Type  Distinct values (train)  Identifier  Constant on train                       Treatment
         ip.src object                       14        True              False           Excluded (identifier)
         ip.dst object                        1        True               True Excluded (identifier; constant)
    tcp.srcport  int64                    30056        True              False           Excluded (identifier)
    tcp.dstport  int64                        1        True               True Excluded (identifier; consta

## 2. Descriptive statistics and correlation (numeric attributes)

In [5]:
descriptive_outputs(X, num_cols, tr, "Supplementary (APA descriptive statistics)", "Supplementary (APA correlation)")

## 3. Mutual-information ranking on all 22 attributes (training data only)

In [6]:
MI = mi_outputs(X, y, tr, num_cols, cat_cols, "APA")
print(MI.to_string(index=False))

        Feature  MI score  Rank  In Top-5  In Top-12
     frame.time  1.039298     1      True       True
    ip.flags.df  0.706154     2      True       True
       tx bytes  0.699934     3      True       True
      frame.len  0.698469     4      True       True
          bytes  0.695481     5      True       True
       rx bytes  0.695201     6     False       True
         ip.src  0.694136     7     False       True
    tcp.srcport  0.680095     8     False       True
     tx packets  0.520838     9     False       True
        packets  0.414868    10     False       True
     rx packets  0.361145    11     False       True
 tcp.flags.push  0.353200    12     False       True
        tcp.seq  0.012040    13     False      False
  tcp.flags.ack  0.010884    14     False      False
    tcp.dstport  0.009331    15     False      False
       ip.proto  0.008740    16     False      False
        tcp.ack  0.006937    17     False      False
tcp.flags.reset  0.002887    18     False     

## 4–6. Six classifiers on Raw-22, MI Top-5 and MI Top-12 (held-out test set)
Raw-22 is the original setting (all attributes, text fields ordinal-encoded on the training set).

In [7]:
RES, PREDS, SELECTED = evaluate(X, y, tr, te, num_cols, cat_cols, models_main(), FEATURE_SETS, "main", FS_DIR)
json.dump(SELECTED, open(p("03_mutual_information", "selected_features.json"), "w"), indent=2)
print("\nSelected features:", json.dumps(SELECTED, indent=1))
save_table(RES.round(6), "04_raw", "test_results_all_feature_sets", "Table 13 source", "Every metric, every model, every feature set")
colsA = ["Model", "Accuracy (%)", "Macro precision (%)", "Macro recall (%)", "Macro specificity (%)", "Macro F1 (%)",
         "Micro F1 (%)", "Weighted F1 (%)", "ROC-AUC (macro OvR)", "PR-AUC (macro OvR)"] + [f"Recall {c} (%)" for c in CLASS_NAMES]
for fs, d in FS_DIR.items():
    t = RES[RES["Feature set"] == fs][colsA].round(4)
    save_table(t, d, f"test_metrics_{fs.replace(' ', '').replace('-', '')}",
               "Table 13" if fs == "RAW" else f"APA test results {fs}", f"Held-out test metrics, {fs}")
    print(f"\n{fs}\n", t.to_string(index=False))
t17 = RES[RES["Feature set"] == "RAW"][["Model", "ROC-AUC (macro OvR)", "Training time mean (s)"]].round(4)
save_table(t17, "04_raw", "auc_training_time_raw", "Table 17", "ROC-AUC and training time, Raw-22")

  [main] RAW        Logistic Regression            acc 100.0000  F1 100.0000  AUC 1.0000  train    0.990s  infer      7.14ms
  [main] RAW        SVM                            acc  99.9669  F1  99.9559  AUC 1.0000  train    1.242s  infer    302.21ms
  [main] RAW        KNN                            acc  99.7266  F1  99.6355  AUC 0.9976  train    0.011s  infer  60635.60ms
  [main] RAW        Decision Tree                  acc  75.0000  F1  55.5556  AUC 0.7778  train    0.140s  infer      3.53ms
  [main] RAW        Random Forest                  acc  76.5675  F1  59.9632  AUC 1.0000  train    0.425s  infer     28.99ms
  [main] RAW        XGBoost                        acc  77.5750  F1  62.5710  AUC 1.0000  train    1.198s  infer     47.99ms
  [main] MI Top-5   Logistic Regression            acc  52.7160  F1  40.9075  AUC 0.8889  train    0.753s  infer      6.45ms
  [main] MI Top-5   SVM                            acc  75.0000  F1  55.5556  AUC 0.8914  train    1.497s  infer   1339.97ms


'/kaggle/working/APA_DDoS_results/04_raw/auc_training_time_raw.csv'

## 7. Raw vs MI Top-5 vs MI Top-12

In [8]:
TA = comparison_outputs(RES, "MI Top-5", "APA Fig (acc/time Top-5)", "APA Fig (F1 Top-5)", "APA Fig (cost Top-5)", "APA Fig (heatmap Top-5)", "APA Table (Raw vs Top-5)")
TB = comparison_outputs(RES, "MI Top-12", "APA Fig (acc/time Top-12)", "APA Fig (F1 Top-12)", "APA Fig (cost Top-12)", "APA Fig (heatmap Top-12)", "APA Table (Raw vs Top-12)")
acc_all = RES.pivot_table(index="Model", columns="Feature set", values="Accuracy (%)", sort=False)[list(FEATURE_SETS)].round(2)
save_table(acc_all, "07_comparison", "accuracy_all_feature_sets", "APA accuracy table", "Accuracy by feature set", index=True)
print(acc_all)

Feature set             RAW  MI Top-5  MI Top-12
Model                                           
Logistic Regression  100.00     52.72     100.00
SVM                   99.97     75.00      99.97
KNN                   99.73     75.00      99.73
Decision Tree         75.00     75.00      75.00
Random Forest         76.57     75.00      95.07
XGBoost               77.57     75.00      77.57


## 8. Stratified 10-fold cross-validation (training portion; SVM excluded for runtime)

In [9]:
if RUN_CV:
    cv_models = {k: v for k, v in models_main().items() if k != "SVM"}
    CV = cross_validate_models(X, y, tr, num_cols, cat_cols, cv_models, FEATURE_SETS, CV_FOLDS)
    save_table(CV.round(4), "08_cross_validation", "cv_all", "APA CV table source", "10-fold CV, all metrics")
    for fs in FEATURE_SETS:
        save_table(cv_paper_table(CV, fs), "08_cross_validation", f"cv_{fs.replace(' ', '').replace('-', '')}",
                   f"APA CV table ({fs})", f"10-fold CV, {fs}")
    cv_figure(CV, list(FEATURE_SETS), "APA CV figure")

  CV RAW        Logistic Regression    100.000 ± 0.000
  CV RAW        KNN                     99.741 ± 0.048
  CV RAW        Decision Tree           75.000 ± 0.000
  CV RAW        Random Forest           82.850 ± 7.314
  CV RAW        XGBoost                 76.334 ± 0.199
  CV MI Top-5   Logistic Regression     56.541 ± 3.120
  CV MI Top-5   KNN                     75.000 ± 0.000
  CV MI Top-5   Decision Tree           75.000 ± 0.000
  CV MI Top-5   Random Forest           75.000 ± 0.000
  CV MI Top-5   XGBoost                 75.000 ± 0.000
  CV MI Top-12  Logistic Regression    100.000 ± 0.000
  CV MI Top-12  KNN                     99.741 ± 0.048
  CV MI Top-12  Decision Tree           75.000 ± 0.000
  CV MI Top-12  Random Forest           98.824 ± 2.415
  CV MI Top-12  XGBoost                 76.334 ± 0.199


## 9. McNemar's test (all pairs, Holm correction)

In [10]:
MCN = mcnemar_all(y_te, PREDS, FEATURE_SETS, list(models_main()))
save_table(MCN, "09_mcnemar", "mcnemar_all_pairs_holm", "APA statistics table", "All 15 pairs per feature set, Holm-adjusted")
print(MCN.to_string(index=False))

Feature set             Model 1       Model 2  b (Model 1 correct, Model 2 wrong)  c (Model 1 wrong, Model 2 correct)    Statistic       p-value                                  Test  p-value (Holm) Significant at 0.05 (Holm)
        RAW Logistic Regression           SVM                                  15                                   0          NaN  6.103516e-05                        exact binomial    6.103516e-05                        Yes
        RAW Logistic Regression           KNN                                 124                                   0   122.008065  2.299015e-28 chi-square with continuity correction    6.897045e-28                        Yes
        RAW Logistic Regression Decision Tree                               11340                                   0 11338.000088  0.000000e+00 chi-square with continuity correction    0.000000e+00                        Yes
        RAW Logistic Regression Random Forest                               10629               

## 10. Timing

In [11]:
timing_outputs(RES, list(FEATURE_SETS), "APA timing table", "APA latency figure", "APA throughput figure")

## 11. Capacity check on Raw-22 (original setting)

In [12]:
UNC, _, _ = evaluate(X, y, tr, te, num_cols, cat_cols, models_unconstrained(), {"RAW": None}, "unconstrained",
                     {"RAW": "11_capacity_check"}, timing=False, figures=False)
save_table(UNC.round(6), "11_capacity_check", "unconstrained_raw", "Table 14 (U rows, E0) / Section 5.2", "Unconstrained models on Raw-22")
print(UNC[["Model", "Accuracy (%)", "Macro F1 (%)"]].round(3).to_string(index=False))

  [unconstrained] RAW        Decision Tree (unconstrained)  acc  75.0000  F1  55.5556  AUC 0.7778  train    0.137s  infer      3.91ms
  [unconstrained] RAW        Random Forest (unconstrained)  acc  76.0692  F1  58.6113  AUC 1.0000  train    3.571s  infer    145.46ms
  [unconstrained] RAW        XGBoost (500 trees, depth 10)  acc  77.5750  F1  62.5710  AUC 1.0000  train    4.642s  infer    194.19ms
                        Model  Accuracy (%)  Macro F1 (%)
Decision Tree (unconstrained)        75.000        55.556
Random Forest (unconstrained)        76.069        58.611
XGBoost (500 trees, depth 10)        77.575        62.571


## 12. Leakage audit
E0 = Raw-22 (Sections 4–6) · E0b = Raw-22 with the timestamp as a number · E1 = identifiers and constants removed ·
E1-Top-5 = MI Top-5 within E1 · E2 = E1 without tcp.flags.push · E3 = E2 features, benign vs attack.

In [13]:
def parse_time(s):
    t = s.astype(str).str.strip().str.replace(r"(\.\d{6})\d+", r"\1", regex=True)
    t = t.str.replace(r"\s+[A-Z]{2,5}$", "", regex=True)
    out = pd.to_datetime(t, format="%d-%b %Y %H:%M:%S.%f", errors="coerce")
    if out.isna().mean() > 0.1:
        out = pd.to_datetime(t, errors="coerce", format="mixed")
    return out

AUD = "12_dataset_specific"
audit_models = models_main() if RUN_SVM_AUDIT else {k: v for k, v in models_main().items() if k != "SVM"}
AUDIT = []

# E0b: timestamp as a number
ts = parse_time(X[TIME_COL]); parsed = ts.notna().mean()
print(f"frame.time parsed: {parsed:.1%}")
if parsed > 0.9:
    X0b = X.copy(); X0b[TIME_COL] = (ts - ts.min()).dt.total_seconds()
    cat0b = [c for c in cat_cols if c != TIME_COL]; num0b = [c for c in X0b.columns if c not in cat0b]
    for md, tag in [(audit_models, "E0b"), (models_unconstrained(), "E0b_unconstrained")]:
        r, _, _ = evaluate(X0b, y, tr, te, num0b, cat0b, md, {"E0b": None}, tag, {"E0b": AUD}, timing=False,
                           figures=(tag == "E0b"))
        AUDIT.append(r)

# E1, E1-Top-5, E2 (numeric features only)
for cols, k, tag in [(CLEAN, None, "E1"), (CLEAN, 5, "E1_Top5"), (E2_COLS, None, "E2")]:
    for md, t2 in [(audit_models, tag), (models_unconstrained(), tag + "_unconstrained")]:
        r, pr, sel = evaluate(X[cols], y, tr, te, list(cols), [], md, {tag: k}, t2, {tag: AUD}, timing=False,
                              figures=(md is audit_models))
        r["Selected features"] = ", ".join(sel[tag]); AUDIT.append(r)

# E3: binary benign vs attack with E2 features
_saved = (N_CLASSES, CLASS_NAMES)
benign_idx = CLASS_NAMES.index([c for c in CLASS_NAMES if "benign" in c.lower()][0])
y3 = (y != benign_idx).astype(int); N_CLASSES, CLASS_NAMES = 2, ["Benign", "Attack"]
for md, t2 in [(models_main() if RUN_SVM_AUDIT else {k: v for k, v in models_main().items() if k != "SVM"}, "E3"),
               (models_unconstrained(), "E3_unconstrained")]:
    r, _, _ = evaluate(X[E2_COLS], y3, tr, te, list(E2_COLS), [], md, {"E3": None}, t2, {"E3": AUD}, timing=False,
                       figures=(t2 == "E3"))
    AUDIT.append(r)
N_CLASSES, CLASS_NAMES = _saved

AUDIT = pd.concat(AUDIT, ignore_index=True)
AUDIT["Experiment"] = AUDIT["Split"].str.replace("_unconstrained", "", regex=False)
AUDIT["Setting"] = np.where(AUDIT["Split"].str.endswith("_unconstrained"), "unconstrained", "constrained")
save_table(AUDIT.round(6), "12_dataset_specific", "audit_all_metrics", "Table 14 source", "All audit experiments, all metrics")

# Table 14: test accuracy per experiment (E0 from Sections 4 and 11)
e0 = pd.concat([RES[RES["Feature set"] == "RAW"].assign(Experiment="E0", Setting="constrained"),
                UNC.assign(Experiment="E0", Setting="unconstrained")])
allx = pd.concat([e0, AUDIT], ignore_index=True)
allx["Model (setting)"] = allx["Model"].str.replace(r" \(unconstrained\)| \(500 trees, depth 10\)", "", regex=True) + \
                          np.where(allx["Setting"] == "unconstrained", " (U)", " (C)")
T14 = allx.pivot_table(index="Model (setting)", columns="Experiment", values="Accuracy (%)", sort=False)
T14 = T14[[c for c in ["E0", "E0b", "E1", "E1_Top5", "E2", "E3"] if c in T14.columns]].round(2)
if "E2" in allx["Experiment"].values:
    f1e2 = allx[allx["Experiment"] == "E2"].set_index("Model (setting)")["Macro F1 (%)"].round(2)
    T14["E2 macro F1"] = f1e2
save_table(T14, "12_dataset_specific", "table14_test_accuracy", "Table 14", "Test accuracy (%) under each configuration", index=True)
print(T14.to_string())

# single-feature tests (depth-1 tree, one attribute at a time; fit on train, score on test)
def stump_table(Xs, ys, pair, label):
    m_tr = np.isin(ys[tr], pair); m_te = np.isin(ys[te], pair); rows = []
    enc = make_pipeline([c for c in Xs.columns if c not in cat_cols], [c for c in Xs.columns if c in cat_cols], None, None)
    Ztr = enc.fit(Xs.iloc[tr], ys[tr]).transform(Xs.iloc[tr]); Zte = enc.transform(Xs.iloc[te])
    names = [c for c in Xs.columns if c not in cat_cols] + [c for c in Xs.columns if c in cat_cols]
    for j, c in enumerate(names):
        st = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(Ztr[m_tr][:, [j]], ys[tr][m_tr])
        rows.append({"Configuration": label, "Feature": c, "Test accuracy (%)": st.score(Zte[m_te][:, [j]], ys[te][m_te]) * 100})
    return pd.DataFrame(rows).sort_values("Test accuracy (%)", ascending=False)

ack = [i for i, c in enumerate(CLASS_NAMES) if "ack" in c.lower() and "psh" not in c.lower()][0]
psh = [i for i, c in enumerate(CLASS_NAMES) if "psh" in c.lower()][0]
STUMP = pd.concat([stump_table(X, y, [ack, psh], "E0: DDoS-ACK vs DDoS-PSH-ACK"),
                   stump_table(X[CLEAN], y, [ack, psh], "E1: DDoS-ACK vs DDoS-PSH-ACK"),
                   stump_table(X[E2_COLS], y, [ack, psh], "E2: DDoS-ACK vs DDoS-PSH-ACK"),
                   stump_table(X[E2_COLS], y3, [0, 1], "E3: benign vs attack")], ignore_index=True).round(3)
save_table(STUMP, "12_dataset_specific", "single_feature_tests", "Section 4.6.3", "Depth-1 tree on one attribute at a time")
print(STUMP.groupby("Configuration").head(4).to_string(index=False))

# ablation figure
fig = fig_grouped_bars(allx[allx["Setting"] == "constrained"].rename(columns={"Feature set": "_fs", "Experiment": "Feature set"}),
                       "Accuracy (%)", "Test accuracy (%)", "APA-DDoS test accuracy by configuration (Section 3.2 settings)",
                       [c for c in ["E0", "E0b", "E1", "E1_Top5", "E2", "E3"] if c in allx["Experiment"].values],
                       annotate=False, legend_title="Configuration")
save_fig(fig, "12_dataset_specific", "fig_ablation_accuracy", "New APA audit figure", "Accuracy per configuration")

frame.time parsed: 0.0%
  [E1] E1         Logistic Regression            acc 100.0000  F1 100.0000  AUC 1.0000  train    0.864s  infer      6.74ms
  [E1] E1         SVM                            acc 100.0000  F1 100.0000  AUC 1.0000  train    0.573s  infer    130.30ms
  [E1] E1         KNN                            acc 100.0000  F1 100.0000  AUC 1.0000  train    0.210s  infer   7528.42ms
  [E1] E1         Decision Tree                  acc 100.0000  F1 100.0000  AUC 1.0000  train    0.032s  infer      2.90ms
  [E1] E1         Random Forest                  acc 100.0000  F1 100.0000  AUC 1.0000  train    0.243s  infer     27.54ms
  [E1] E1         XGBoost                        acc 100.0000  F1 100.0000  AUC 1.0000  train    0.869s  infer     50.88ms
  [E1_unconstrained] E1         Decision Tree (unconstrained)  acc 100.0000  F1 100.0000  AUC 1.0000  train    0.031s  infer      2.86ms
  [E1_unconstrained] E1         Random Forest (unconstrained)  acc 100.0000  F1 100.0000  AUC 1.0000 

'/kaggle/working/APA_DDoS_results/12_dataset_specific/fig_ablation_accuracy.png'

## 13. Export

In [15]:
export_all()


Exported 26 tables and 74 figures.
Paper map: /kaggle/working/APA_DDoS_results/13_export/paper_map.csv
Zip for download: /kaggle/working/APA_DDoS_results.zip
